# Narrative Similarity Retrieval Pipeline

**Thesis**: Quantifying the Impact of Temporal and Causal Structures on Narrative Similarity Retrieval

This notebook implements the full methodology pipeline:

1. **Dataset subsetting** - Stratified sample of Tell Me Again! based on EDA findings
2. **BERT+CRF training** - Train event detection model on MAVEN
3. **Event extraction** - Extract `(trigger_span, event_type)` pairs from narrative summaries
4. **Temporal annotation** - Llama-3-8B zero-shot inference for temporal relations
5. **Causal annotation** - Llama-3-8B zero-shot inference for causal relations
6. **Joint temporo-causal annotation** - Llama-3-8B zero-shot inference for joint relations
7. **Narrative linearization** - Convert event graphs to text per experimental condition
8. **Embedding** - Encode with E5 and Sentence-BERT
9. **Baselines** - BoW, TF-IDF, StoryEmbed
10. **Evaluation** - Retrieval metrics, statistical significance, robustness analysis

Each step caches its output to `data/intermediate/` so expensive computation is never repeated.
Reusable code is extracted to `src/` as needed.

## 0. Setup & Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import json
import pickle
from tell_me_again import StoryDataset
from transformers import AutoTokenizer

# Project root (one level up from notebooks/)
ROOT = Path(os.getcwd()).parent
DATA_RAW = ROOT / "data" / "raw"
DATA_INTERMEDIATE = ROOT / "data" / "intermediate"
DATA_RESULTS = ROOT / "data" / "results"

# Create intermediate/results dirs if they don't exist
DATA_INTERMEDIATE.mkdir(parents=True, exist_ok=True)
DATA_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Raw data:     {DATA_RAW}")
print(f"Intermediate: {DATA_INTERMEDIATE}")
print(f"Results:      {DATA_RESULTS}")

## 1. Event Detection (MAVEN)

### 1.1 Load & Explore MAVEN

MAVEN contains 4,480 Wikipedia documents annotated with 168 event types. Each document has:
- `content`: list of sentences with tokenized words
- `events`: list of event types, each with one or more trigger mentions (word + sentence + offset)
- `negative_triggers`: words that look like events but aren't

Splits: 2,913 train / 710 valid / 857 test documents.

In [ ]:
def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    with open(path, "r") as f:
        return [json.loads(line) for line in f]

# Load MAVEN data
MAVEN_DIR = DATA_RAW / "MAVEN"
train_maven = load_jsonl(MAVEN_DIR / "train.jsonl")
valid_maven = load_jsonl(MAVEN_DIR / "valid.jsonl")
test_maven = load_jsonl(MAVEN_DIR / "test.jsonl")

print(f"Train: {len(train_maven)} docs")
print(f"Valid: {len(valid_maven)} docs")
print(f"Test:  {len(test_maven)} docs")

In [ ]:
# Quick look at a single document
doc = train_maven[0]
print(f"Title: {doc['title']}")
print(f"Sentences: {len(doc['content'])}")
print(f"Event types: {len(doc['events'])}")
print(f"Negative triggers: {len(doc['negative_triggers'])}")

# Print details of the first 3 events of the first document
for i, evt in enumerate(doc["events"][:3], start=1):
    print(f"\nEvent {i}:")
    print(f"  Type: {evt['type']} (id: {evt['type_id']})")
    for mention in evt["mention"]:
        print(f"  Trigger: '{mention['trigger_word']}' in sentence {mention['sent_id']}, offset {mention['offset']}")


### 1.2 Train BERT+CRF

The training code (`models/bert_crf/`) is based on the [THU-KEG MAVEN baseline](https://github.com/THU-KEG/MAVEN-dataset/tree/main/baselines/BERT%2BCRF). The original code targeted `transformers==2.6.0` / `torch==1.2.0`, but those versions have no arm64 (Apple Silicon) wheels. We use `transformers==4.18.0` + `tokenizers==0.11.6` (oldest compatible combination with arm64 wheels) and patched 3 lines (see `PATCHED` comments in `bert_crf.py` and `run_maven.py`). No training logic was changed.

> **Alternative — one-shot reproducible setup + training:** everything below is wrapped in `models/bert_crf/train_bert_crf_on_maven.sh`. From the project root, run:
>
> ```bash
> bash models/bert_crf/train_bert_crf_on_maven.sh
> ```
>
> The script detects the OS (macOS / Linux / Windows via Git Bash or WSL), locates a Python 3.9 interpreter, creates the colocated venv, installs the pinned dependencies, checks the MAVEN data is in place, then trains + evaluates. All stdout/stderr is tee'd to `data/intermediate/models/bert_crf/logs/train_<timestamp>.log` so every run is preserved. Use `--setup-only` to stop after `pip install`, or `--eval-only` to re-run evaluation against an existing checkpoint.
>
> The manual steps below do the same thing and remain the source of truth for what the script does.

**Important pinning notes**:

- The venv is created **inside `models/bert_crf/`** (not at project root) so it lives next to the code that imports it and cannot be shadowed by another venv on PATH.
- Python must be **3.9**. `transformers==4.18.0` has no wheels for Python 3.11+, so creating the venv with a newer interpreter will silently pull a newer transformers release, and the custom `BertCRFForTokenClassification` subclass will break against transformers 5.x internals (`all_tied_weights_keys` attribute error). We invoke `/usr/bin/python3` explicitly because Apple's CommandLineTools still ship Python 3.9.6 at that path on macOS 14/15; this avoids any Homebrew/pyenv interpreter that may be earlier on `PATH`.

**Setup (from project root, in terminal):**

```bash
cd models/bert_crf

# Create the venv with Apple's system Python 3.9 (colocated with the code)
/usr/bin/python3 -m venv .venv-maven-train
source .venv-maven-train/bin/activate

# Verify before installing — abort if this is not 3.9.x
python -V

pip install --upgrade pip
pip install torch transformers==4.18.0 tokenizers==0.11.6 seqeval scikit-learn numpy
```

**Train (same terminal, still inside `models/bert_crf/`):**

```bash
python run_maven.py \
    --data_dir ../../data/raw/MAVEN/ \
    --model_type bertcrf \
    --model_name_or_path bert-base-uncased \
    --output_dir ../../data/intermediate/models/bert_crf/ \
    --max_seq_length 128 \
    --do_lower_case \
    --per_gpu_train_batch_size 16 \
    --per_gpu_eval_batch_size 16 \
    --gradient_accumulation_steps 8 \
    --learning_rate 5e-5 \
    --num_train_epochs 5 \
    --save_steps 100 \
    --logging_steps 100 \
    --seed 0 \
    --do_train \
    --do_eval \
    --evaluate_during_training \
    --overwrite_output_dir
```

**Final evaluation against `valid.jsonl`** (runs automatically at the end of the command above; can also be re-run standalone using the saved checkpoint):

```bash
python run_maven.py \
    --data_dir ../../data/raw/MAVEN/ \
    --model_type bertcrf \
    --model_name_or_path bert-base-uncased \
    --output_dir ../../data/intermediate/models/bert_crf/ \
    --max_seq_length 128 \
    --do_lower_case \
    --per_gpu_eval_batch_size 16 \
    --seed 0 \
    --do_eval
```

This writes `data/intermediate/models/bert_crf/eval_results.txt` with `f1`, `precision`, `recall`, `loss` on the MAVEN validation split.

```bash
deactivate  # when done
```

After training completes, the checkpoint directory will contain:
- `pytorch_model.bin` — model weights
- `config.json` — BERT config with num_labels=337 (168 event types × 2 for B/I tags + O)
- `vocab.txt` — BERT tokenizer vocabulary
- `eval_results.txt` — final metrics on `valid.jsonl`

In [ ]:
# Define the path to the BERT-CRF checkpoint (from Section 1.2)
BERT_CRF_CHECKPOINT = DATA_INTERMEDIATE / "models" / "bert_crf" 

# Verify checkpoint exists
required_files = ["pytorch_model.bin", "config.json", "vocab.txt"]
isCheckPointMissing = [f for f in required_files if not (BERT_CRF_CHECKPOINT / f).exists()]

# Print checkpoint status and file sizes
if isCheckPointMissing:
    print(f"Checkpoint not found at {BERT_CRF_CHECKPOINT}")
    print(f"Missing files: {isCheckPointMissing}")
    print("Please run the training commands from Section 1.2 first.")
else:
    print(f"Checkpoint found at {BERT_CRF_CHECKPOINT}")
    for f in required_files:
        size_mb = (BERT_CRF_CHECKPOINT / f).stat().st_size / 1e6
        print(f"  {f}: {size_mb:.1f} MB")

## 2. Dataset Subsetting

Build the eligible Tell Me Again! retrieval corpus by applying the EDA-derived filter cascade to the official **train split** (which §3.1.4 of the thesis showed is itself a representative subset of the full corpus).

The output is a flat JSONL where **each row is one summary**, keyed by `wikidata_id` (the relevance-cluster key for retrieval evaluation). Downstream sections append `events`, `temporal_relations`, `causal_relations`, and `embedding` to each row.

Down-Sampling the Full TMA Dataset:
1. Restrict to train split.
2. Drop stories with no Wikidata genre.
3. Drop near-duplicate translated summaries (cosine > `DEDUP_COSINE_THRESHOLD` against EN original) and summaries shorter than `MIN_SENTENCES`.
4. Drop summaries above `MAX_TOKENS` (Llama-3-8B context window).
5. Drop stories left with fewer than `MIN_DEDUPED_EN_SUMMARIES` surviving English summaries.
6. Optionally cap at `N_STORIES` for local development.

Output: `data/intermediate/tma_subset.jsonl`.

In [ ]:
# Dataset Subsetting parameters
N_STORIES: int | None = 5            # keep it short for local dev; None = full eligible set
MIN_SENTENCES: int = 8               # drop summaries shorter than this
MAX_TOKENS: int = 8192               # Llama-3-8B context window
MIN_DEDUPED_EN_SUMMARIES: int = 2    # min surviving EN summaries per story (relevance clusters need >=2)
DEDUP_COSINE_THRESHOLD: float = 0.6  # translation-dedup threshold (matches EDA & prior work)

# Tokenizer: load once and reuse for all summaries (Llama-3-8B-Instruct tokenizer)
LLAMA_TOKENIZER_ID: str = "meta-llama/Meta-Llama-3-8B-Instruct"
# Output path for the final TMA subset after all filtering steps
TMA_SUBSET_PATH = DATA_INTERMEDIATE / "tma_subset.jsonl"

# Print the subsetting parameters for verification
print(f"N_STORIES               = {N_STORIES}")
print(f"MIN_SENTENCES           = {MIN_SENTENCES}")
print(f"MAX_TOKENS              = {MAX_TOKENS}")
print(f"MIN_DEDUPED_EN_SUMMARIES= {MIN_DEDUPED_EN_SUMMARIES}")
print(f"DEDUP_COSINE_THRESHOLD  = {DEDUP_COSINE_THRESHOLD}")
print(f"Output                  -> {TMA_SUBSET_PATH}")

In [ ]:
# Define the path to the raw TellMeAgain zip and the intermediate cache for parsed StoryDataset
TMA_ZIP_PATH = DATA_RAW / "TellMeAgain" / "tell_me_again_v1.zip"
TMA_PARSED_CACHE = DATA_INTERMEDIATE / "_cache_storydataset.pkl"

# Tokenizer: load once and reuse for all summaries (Llama-3-8B-Instruct tokenizer)
if "_tokenizer" not in globals():
    print("Loading Llama-3 tokenizer...")
    _tokenizer = AutoTokenizer.from_pretrained(LLAMA_TOKENIZER_ID)

# Helper function to count Llama tokens in a text
def n_llama_tokens(text: str) -> int:
    return len(_tokenizer.encode(text, add_special_tokens=False))

# In-memory cache + disk-pickle fallback for the parsed StoryDataset, which is expensive to parse from the zip each time.
# We want to reuse it across pipeline iterations while tuning subsetting parameters
if "_ds" not in globals():
    if TMA_PARSED_CACHE.exists():
        print(f"Loading StoryDataset from disk cache ({TMA_PARSED_CACHE.name})...")
        with open(TMA_PARSED_CACHE, "rb") as f:
            _ds = pickle.load(f)
    else:
        print("Parsing TellMeAgain zip...")
        _ds = StoryDataset(data_path=TMA_ZIP_PATH)
        TMA_PARSED_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with open(TMA_PARSED_CACHE, "wb") as f:
            pickle.dump(_ds, f)
        print(f"  -> cached to {TMA_PARSED_CACHE} "
              f"({TMA_PARSED_CACHE.stat().st_size / 1e6:.1f} MB)")

# Restrict the dataset to the train split
if "_train_stories" not in globals():
    _train_stories = _ds.perform_splits()["train"]
print(f"  [1] train split:                       {len(_train_stories):>6} stories")

# Drop stories with no Wikidata genre
train_with_genre = {wid: s for wid, s in _train_stories.stories.items() if s.genres}
print(f"  [2] with >=1 Wikidata genre:           {len(train_with_genre):>6} stories")

# Per-story: dedup translations + min_sentences (handled by package), then per-summary token filter, then min surviving EN summaries.
n_stories_kept = 0
n_summaries_kept = 0
n_summaries_dropped_tokens = 0
rows = []  # buffer; written to JSONL at the end

# Deterministic ordering for the N_STORIES cap
sorted_wids = sorted(train_with_genre.keys())

for wid in sorted_wids:
    story = train_with_genre[wid]
    ids, summaries = story.get_all_summaries_en(
        max_similarity=DEDUP_COSINE_THRESHOLD,
        min_sentences=MIN_SENTENCES,
    )

    surviving = []
    for summary_id, text in zip(ids, summaries):
        n_tok = n_llama_tokens(text)
        if n_tok > MAX_TOKENS:
            n_summaries_dropped_tokens += 1
            continue
        sentences = story.sentences[summary_id]
        surviving.append({
            "wikidata_id": wid,
            "summary_id": summary_id,
            "lang": summary_id,                # source-language code; text is English in all cases
            "text": text,
            "sentences": sentences,
            "n_sentences": len(sentences),
            "n_tokens": n_tok,
            "genres": story.genres,
            "split": "train",
        })

    if len(surviving) < MIN_DEDUPED_EN_SUMMARIES:
        continue

    n_stories_kept += 1
    n_summaries_kept += len(surviving)
    rows.extend(surviving)

    if N_STORIES is not None and n_stories_kept >= N_STORIES:
        break

print(f"  [3-5] after dedup + length + cluster:  {n_stories_kept:>6} stories, {n_summaries_kept} summaries")
print(f"        ({n_summaries_dropped_tokens} summaries dropped for >{MAX_TOKENS} tokens)")

# 6. Stream-write JSONL
TMA_SUBSET_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(TMA_SUBSET_PATH, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"\n  Wrote {len(rows)} summaries across {n_stories_kept} stories")
print(f"       -> {TMA_SUBSET_PATH}")
print(f"       size: {TMA_SUBSET_PATH.stat().st_size / 1024:.1f} KB")